# Step 6b — GAN Augmentation vs Baseline Anchor (C1 test)

**RESS 2025 — GAN-Conformal-RUL**

Tests the central hypothesis: does augmenting the training set with synthetic near-failure
windows improve **near-failure RUL RMSE**? An initial run at `augment_ratio=1.0` with kNN labels
made near-failure RMSE *worse* in 4/5 folds, and the synthetic RUL labels skewed high
(mean 0.27 vs real 0.24). This notebook diagnoses that rather than concluding from one setting:

1. **Baseline anchor** — real data only (piecewise RUL, no masking).
2. **Ratio sweep** — augment_ratio in {0.25, 0.5, 0.75, 1.0}.
3. **Labeling method** — kNN-borrowed vs independent-sample from real S3 RUL.

The deliverable is a table of near-failure RMSE (and overall) across all settings, so the
decision to keep, tune, or drop C1 rests on evidence.

> **Interface note:** the augmentation call is wrapped in `augment_fold()` near the top. If your
> `gan.augment()` signature differs, fix it there once — everything else calls the wrapper.

## 1. Setup

In [ ]:
import os, sys, shutil
os.chdir('/content')
REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH); sys.path.insert(0, f'{REPO_PATH}/src')
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = f'{REPO_PATH}/figures'; os.makedirs(SAVE_DIR, exist_ok=True)
CKPT_DIR = '/content/drive/MyDrive/ress_checkpoints'
import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import prepare_all_folds, build_folds, WINDOW_SIZE
from model import ModelConfig, RULTrainer
from gan import StageGAN, GANConfig

TARGET = 2   # 0-indexed near-failure stage

## 2. Rebuild data (piecewise RUL, log_clip) + load frozen GANs

In [ ]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_ROOT, f'not found: {CANDIDATES}'

all_data = XJTUSYLoader(DATA_ROOT).load_all()
pipeline = HealthIndicatorPipeline(fpt_consecutive=5, fpt_min_relative_rise=0.20)
results = pipeline.process_all(all_data, verbose=False)

# piecewise target is now the default in windowing.py
fold_data = prepare_all_folds(results, scaling_method='log_clip',
                              rul_target='piecewise', verbose=False)
folds = build_folds(results)
print(f'{len(fold_data)} folds ready (piecewise RUL).')

In [ ]:
# Load the 5 frozen generators from Drive
gans = {}
for k in range(1, 6):
    path = f'{CKPT_DIR}/gan_fold{k}.pt'
    assert os.path.exists(path), f'missing {path}'
    ckpt = torch.load(path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    cfg = GANConfig(**ckpt['config'])
    g = StageGAN(cfg)
    g.G.load_state_dict(ckpt['generator'])
    g.D.load_state_dict(ckpt['critic'])
    gans[k] = g
print(f'{len(gans)}/5 GANs loaded.')

## 3. Augmentation helper

Generates synthetic near-failure windows and assigns each a RUL label by one of two methods:

- **`knn`**: borrow the RUL of the nearest real S3 window in feature space (weak content-label tie).
- **`sample`**: draw independently from the real S3 RUL distribution (guarantees the marginal matches).

This wrapper is self-contained so it doesn't depend on `gan.augment()`'s exact signature; it uses
`gan.sample(n, stage)` which we already rely on for the quality gates.

In [ ]:
def augment_fold(gan, d, ratio, label_method='sample', seed=0):
    """Return (X_aug, y_rul_aug, y_stage_aug) = real train + synthetic S3.

    ratio: n_synthetic / n_real_S3.
    label_method: 'knn' | 'sample'.
    """
    rng = np.random.default_rng(seed)
    Xtr, ytr, str_ = d['X_train'], d['y_rul_train'], d['y_stage_train']

    s3 = str_ == TARGET
    X_s3, y_s3 = Xtr[s3], ytr[s3]
    n_real = len(X_s3)
    n_syn = int(round(ratio * n_real))
    if n_syn == 0:
        return Xtr, ytr, str_

    X_syn = gan.sample(n_syn, TARGET)                    # (n_syn, 32, 20)

    if label_method == 'knn':
        tree = cKDTree(X_s3.reshape(n_real, -1))
        _, idx = tree.query(X_syn.reshape(n_syn, -1), k=1)
        y_syn = y_s3[idx]
    elif label_method == 'sample':
        y_syn = rng.choice(y_s3, size=n_syn, replace=True)
    else:
        raise ValueError(label_method)

    stage_syn = np.full(n_syn, TARGET, dtype=str_.dtype)
    X_aug = np.concatenate([Xtr, X_syn.astype(np.float32)])
    y_aug = np.concatenate([ytr, y_syn.astype(np.float32)])
    s_aug = np.concatenate([str_, stage_syn])
    print(f"    aug ratio={ratio} method={label_method}: real S3={n_real} "
          f"+{n_syn} syn | RUL syn mean {y_syn.mean():.3f} vs real {y_s3.mean():.3f}")
    return X_aug, y_aug, s_aug

## 4. Train-and-evaluate helper

Trains one fold on given (possibly augmented) data and returns per-stage test RMSE. Masking is
off because the piecewise target makes healthy windows learnable; synthetic windows are stage-3
and carry RUL labels, so they participate in regression as intended.

In [ ]:
def train_eval(d, X_tr, y_tr, s_tr, cfg):
    tr = RULTrainer(cfg)
    tr.fit(X_tr, y_tr, d['X_val'], d['y_rul_val'],
           stage_train=s_tr, stage_val=d['y_stage_val'],
           mask_healthy=False, verbose=False)
    yp = tr.predict(d['X_test']); yt = d['y_rul_test']; st = d['y_stage_test']
    out = {'overall': float(np.sqrt(np.mean((yp - yt) ** 2)))}
    for s, name in [(0, 'healthy'), (1, 'early'), (2, 'nearfail')]:
        m = st == s
        out[name] = float(np.sqrt(np.mean((yp[m] - yt[m]) ** 2))) if m.sum() else np.nan
    return out

## 5. Baseline anchor (real data only)

In [ ]:
cfg = ModelConfig(epochs=100, patience=25, lr=5e-4)

base = {}
print('BASELINE (real only)')
for d in fold_data:
    k = d['fold']
    r = train_eval(d, d['X_train'], d['y_rul_train'], d['y_stage_train'], cfg)
    base[k] = r
    print(f"  fold {k}: nearfail {r['nearfail']:.4f} | overall {r['overall']:.4f}")
base_nf = np.mean([base[k]['nearfail'] for k in base])
print(f"  MEAN near-failure RMSE: {base_nf:.4f}")

## 6. Sweep: ratio x labeling method

For each (ratio, method) we train all 5 folds and record mean near-failure RMSE. This is the
table that decides C1. ~5 folds x 4 ratios x 2 methods = 40 short trainings; a few minutes each
fold on the T4, so budget ~30-40 min. Reduce `RATIOS` or `METHODS` to shorten.

In [ ]:
RATIOS = [0.25, 0.5, 0.75, 1.0]
METHODS = ['sample', 'knn']

sweep = []
for method in METHODS:
    for ratio in RATIOS:
        print(f"\n=== method={method}  ratio={ratio} ===")
        nf, ov = [], []
        for d in fold_data:
            k = d['fold']
            Xa, ya, sa = augment_fold(gans[k], d, ratio, label_method=method, seed=k)
            r = train_eval(d, Xa, ya, sa, cfg)
            nf.append(r['nearfail']); ov.append(r['overall'])
        sweep.append({'method': method, 'ratio': ratio,
                      'nearfail': np.mean(nf), 'overall': np.mean(ov)})
        print(f"  -> mean nearfail {np.mean(nf):.4f} | overall {np.mean(ov):.4f} "
              f"(baseline nf {base_nf:.4f})")

## 7. Decision table

In [ ]:
sw = pd.DataFrame(sweep)
sw['nf_vs_base'] = sw['nearfail'] - base_nf   # negative = improvement
print(f"Baseline mean near-failure RMSE: {base_nf:.4f}\n")
print(sw.to_string(index=False,
      formatters={'nearfail':'{:.4f}'.format, 'overall':'{:.4f}'.format,
                  'nf_vs_base':'{:+.4f}'.format}))
best = sw.loc[sw['nearfail'].idxmin()]
print(f"\nBest setting: method={best['method']} ratio={best['ratio']} "
      f"-> near-failure {best['nearfail']:.4f} ({best['nf_vs_base']:+.4f} vs baseline)")
if best['nearfail'] < base_nf:
    print('C1 HELPS at the best setting — report this ratio/method.')
else:
    print('C1 does not improve near-failure RMSE at any setting tested.')
    print('Pivot: test whether augmentation still narrows conformal intervals (C3).')

In [ ]:
# Visualise: near-failure RMSE vs ratio, one line per method, baseline as dashed
fig, ax = plt.subplots(figsize=(7, 4.5))
for method in METHODS:
    s = sw[sw['method'] == method].sort_values('ratio')
    ax.plot(s['ratio'], s['nearfail'], marker='o', label=f'{method}')
ax.axhline(base_nf, ls='--', color='gray', label='baseline (real only)')
ax.set_xlabel('augment_ratio'); ax.set_ylabel('mean near-failure RMSE')
ax.set_title('Does GAN augmentation help near-failure RUL?')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/06b_augmentation_sweep.png', dpi=150, bbox_inches='tight')
plt.show()